# Module 02: Pandas for Machine Learning
## Notebook 04: Grouping, Aggregations, and Pivot Tables

In machine learning, aggregated group statistics are among the most powerful features you can engineer. Grouping allows you to summarize historical behavior, calculate cohort metrics, and generate comparative features (e.g., how an individual's spending compares to their demographic average).

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Apply the **Split-Apply-Combine** pattern using `df.groupby()`.
2. Compute multi-metric aggregations across features using `.agg()`.
3. Compute group-level normalization and feature offsets using `.transform()`.
4. Reshape data into multi-dimensional summaries using `.pivot_table()`.
5. Compute joint frequency distributions with `pd.crosstab()`.

In [1]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.6


### 1. The Split-Apply-Combine Pattern

When calling `df.groupby('Category')`:
1. **Split**: The dataset is divided into distinct subsets based on grouping keys.
2. **Apply**: An aggregation function (`mean`, `sum`, `std`, `count`) is applied independently to each subset.
3. **Combine**: The individual results are merged back into a single summary DataFrame.

In [2]:
# Simulated e-commerce transaction dataset
sales_data = {
    'Department': ['Electronics', 'Clothing', 'Electronics', 'Home', 'Clothing', 'Electronics', 'Home'],
    'Region': ['North', 'North', 'South', 'South', 'North', 'North', 'South'],
    'Revenue': [1200.0, 350.0, 850.0, 420.0, 600.0, 2100.0, 510.0],
    'Units_Sold': [4, 7, 2, 5, 12, 6, 8]
}

df_sales = pd.DataFrame(sales_data)
print("Sales Dataset:\n", df_sales)

# Basic GroupBy: Total revenue per department
dept_revenue = df_sales.groupby('Department')['Revenue'].sum()
print("\nTotal Revenue per Department:\n", dept_revenue)

Sales Dataset:
     Department Region  Revenue  Units_Sold
0  Electronics  North   1200.0           4
1     Clothing  North    350.0           7
2  Electronics  South    850.0           2
3         Home  South    420.0           5
4     Clothing  North    600.0          12
5  Electronics  North   2100.0           6
6         Home  South    510.0           8

Total Revenue per Department:
 Department
Clothing        950.0
Electronics    4150.0
Home            930.0
Name: Revenue, dtype: float64


---
### 2. Multi-Metric Aggregations with `.agg()`

In exploratory data analysis and feature engineering, we often need multiple summary statistics simultaneously for different features.

In [3]:
# Compute custom metrics per Department and Region
summary_agg = df_sales.groupby(['Department', 'Region']).agg({
    'Revenue': ['count', 'mean', 'max'],
    'Units_Sold': ['sum', 'mean']
}).round(2)

print("Multi-Feature Aggregation Summary:\n", summary_agg)

Multi-Feature Aggregation Summary:
                    Revenue                 Units_Sold     
                     count    mean     max        sum mean
Department  Region                                        
Clothing    North        2   475.0   600.0         19  9.5
Electronics North        2  1650.0  2100.0         10  5.0
            South        1   850.0   850.0          2  2.0
Home        South        2   465.0   510.0         13  6.5


---
### 3. Feature Engineering with `.transform()`

> **Key Difference between `.agg()` and `.transform()`:**
> - `.agg()` collapses rows, returning a smaller summary table.
> - `.transform()` returns an array with the **exact same length as the original DataFrame**, broadcasting the group statistic back to each individual row!
> This allows immediate feature engineering (e.g. comparing a transaction to its department average).

In [4]:
# Compute Department Mean Revenue for every row
df_sales['Dept_Avg_Revenue'] = df_sales.groupby('Department')['Revenue'].transform('mean')

# Calculate relative expenditure ratio (Feature for anomaly detection)
df_sales['Revenue_to_Dept_Avg_Ratio'] = (df_sales['Revenue'] / df_sales['Dept_Avg_Revenue']).round(2)

print("DataFrame with Group-Transformed Features:\n", df_sales[['Department', 'Revenue', 'Dept_Avg_Revenue', 'Revenue_to_Dept_Avg_Ratio']])

DataFrame with Group-Transformed Features:
     Department  Revenue  Dept_Avg_Revenue  Revenue_to_Dept_Avg_Ratio
0  Electronics   1200.0       1383.333333                       0.87
1     Clothing    350.0        475.000000                       0.74
2  Electronics    850.0       1383.333333                       0.61
3         Home    420.0        465.000000                       0.90
4     Clothing    600.0        475.000000                       1.26
5  Electronics   2100.0       1383.333333                       1.52
6         Home    510.0        465.000000                       1.10


---
### 4. Pivot Tables

A `pivot_table` reshapes data into a grid format where:
- `index`: Specifies row categories.
- `columns`: Specifies column categories.
- `values`: The numerical data to aggregate.
- `aggfunc`: The aggregation function (default is `'mean'`).

In [5]:
pivot_rev = df_sales.pivot_table(
    values='Revenue',
    index='Department',
    columns='Region',
    aggfunc='sum',
    fill_value=0.0
)

print("Pivot Table: Total Revenue by Department and Region:\n", pivot_rev)

Pivot Table: Total Revenue by Department and Region:
 Region        North  South
Department                
Clothing      950.0    0.0
Electronics  3300.0  850.0
Home            0.0  930.0


---
### 5. Cross-Tabulations: `pd.crosstab()`

`pd.crosstab()` computes a frequency table of two or more categorical factors.
This is essential in classification problems to inspect class balance across segments or construct confusion matrices.

In [6]:
# Joint frequency distribution of Department and Region
cross_tab = pd.crosstab(df_sales['Department'], df_sales['Region'], margins=True)
print("Frequency Cross-Tabulation:\n", cross_tab)

# Normalized proportions (conditional distribution)
cross_tab_norm = pd.crosstab(df_sales['Department'], df_sales['Region'], normalize='index').round(3)
print("\nRow-Normalized Proportions:\n", cross_tab_norm)

Frequency Cross-Tabulation:
 Region       North  South  All
Department                    
Clothing         2      0    2
Electronics      2      1    3
Home             0      2    2
All              4      3    7

Row-Normalized Proportions:
 Region       North  South
Department               
Clothing     1.000  0.000
Electronics  0.667  0.333
Home         0.000  1.000


### Summary & Next Steps
In this notebook, you mastered:
- The Split-Apply-Combine mechanics of `groupby()`.
- Multi-metric `.agg()` summaries.
- Broadcasting group statistics back to observations using `.transform()`.
- Building two-dimensional summaries with `pivot_table` and `crosstab`.

**Next Notebook:** `05_combining_datasets_and_timeseries.ipynb` — Merge relational tables, concatenate batches, and engineer time series lag features.